In [2]:
import requests
import json
import os
from pathlib import Path
from PIL import Image
import fitz
import math
import io
import time

In [3]:
def extract_text_from_image(file, api_key, model, task_type, max_tokens, temperature, top_p, repetition_penalty, pages=None):
    url = "https://api.opentyphoon.ai/v1/ocr"
    files = {'file': file}
    data = {
        'model': model,
        'task_type': task_type,
        'max_tokens': str(max_tokens),
        'temperature': str(temperature),
        'top_p': str(top_p),
        'repetition_penalty': str(repetition_penalty)
    }
    if pages:
        data['pages'] = json.dumps(pages)
    headers = {
        'Authorization': f'Bearer {api_key}'
    }
    response = requests.post(url, files=files, data=data, headers=headers)
    if response.status_code == 200:
        result = response.json()
        # Extract text from successful results
        extracted_texts = []
        for page_result in result.get('results', []):
            if page_result.get('success') and page_result.get('message'):
                content = page_result['message']['choices'][0]['message']['content']
                try:
                    # Try to parse as JSON if it's structured output
                    parsed_content = json.loads(content)
                    text = parsed_content.get('natural_text', content)
                except json.JSONDecodeError:
                    text = content
                extracted_texts.append(text)
            elif not page_result.get('success'):
                print(f"Error processing {page_result.get('filename', 'unknown')}: {page_result.get('error', 'Unknown error')}")
        return '\n'.join(extracted_texts)
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
        return None

In [4]:
api_key = os.environ["TOKEN_KEY"]
image_path = ""  # or path/to/your/document.pdf
model = "typhoon-ocr"
task_type = "default"
max_tokens = 16384
temperature = 0.1
top_p = 0.6
repetition_penalty = 1.2
pages = []

In [5]:
dir = Path("page-ocr")
files = list(dir.glob("**/*.txt"))

In [6]:
pdf_dir = Path("ลำปาง")
pdf_files = list(pdf_dir.glob("**/*.pdf"))
n2file = dict()
for p_f in pdf_files:
    n2file[p_f.stem] = p_f

In [12]:
for f in pdf_files:
    print(str(f.parent)[6:])

นอกเขต\ชุดที่ 1
นอกเขต\ชุดที่ 1
นอกเขต\ชุดที่ 10
นอกเขต\ชุดที่ 10
นอกเขต\ชุดที่ 11
นอกเขต\ชุดที่ 11
นอกเขต\ชุดที่ 12
นอกเขต\ชุดที่ 12
นอกเขต\ชุดที่ 13
นอกเขต\ชุดที่ 13
นอกเขต\ชุดที่ 2
นอกเขต\ชุดที่ 2
นอกเขต\ชุดที่ 3
นอกเขต\ชุดที่ 3
นอกเขต\ชุดที่ 4
นอกเขต\ชุดที่ 4
นอกเขต\ชุดที่ 5
นอกเขต\ชุดที่ 5
นอกเขต\ชุดที่ 6
นอกเขต\ชุดที่ 6
นอกเขต\ชุดที่ 7
นอกเขต\ชุดที่ 7
นอกเขต\ชุดที่ 8
นอกเขต\ชุดที่ 8
นอกเขต\ชุดที่ 9
นอกเขต\ชุดที่ 9
อำเภองาว\แบบ ส.ส.5-18
อำเภองาว\แบบ ส.ส.5-18
อำเภองาว\แบบ ส.ส.5-18
อำเภองาว\แบบ ส.ส.5-18
อำเภองาว\แบบ ส.ส.5-18
อำเภองาว\แบบ ส.ส.5-18
อำเภองาว\แบบ ส.ส.5-18
อำเภองาว\แบบ ส.ส.5-18
อำเภองาว\แบบ ส.ส.5-18
อำเภองาว\แบบ ส.ส.5-18
อำเภองาว\แบบ ส.ส.5-18 (บช)
อำเภองาว\แบบ ส.ส.5-18 (บช)
อำเภองาว\แบบ ส.ส.5-18 (บช)
อำเภองาว\แบบ ส.ส.5-18 (บช)
อำเภองาว\แบบ ส.ส.5-18 (บช)
อำเภองาว\แบบ ส.ส.5-18 (บช)
อำเภองาว\แบบ ส.ส.5-18 (บช)
อำเภองาว\แบบ ส.ส.5-18 (บช)
อำเภองาว\แบบ ส.ส.5-18 (บช)
อำเภองาว\แบบ ส.ส.5-18 (บช)
อำเภอวังเหนือ\แบบ ส.ส.5-18
อำเภอวังเหนือ\แบบ ส.ส.5-18
อำเภอวังเหนือ\แบบ ส.ส.5-18
อำเภอ

In [7]:
with open("except_ocr.txt", "r") as fe:
    fex = [line.strip() for line in fe]

In [8]:
fex

[]

In [14]:
for file in files:
    path_dir = str(n2file[file.stem].parent)[6:]
    result_base_dir = os.path.join("ocr-result", path_dir)
    success_base_dir = os.path.join("ocr-success-page", path_dir)
    
    os.makedirs(result_base_dir, exist_ok=True)
    os.makedirs(success_base_dir, exist_ok=True)
    with open(file, 'r') as f:
        lines = [int(line.strip()) for line in f]
    lines_check = []
    success_path = f"ocr-success-page/{path_dir}/{file.stem}.txt"
    if os.path.exists(success_path):
        with open(success_path, "r", encoding="utf-8") as ft:
            lines_check = [int(line.strip()) for line in ft if line.strip()]
    pages = lines
    image_path = file.parent / (file.stem + ".txt")
    pf_name = (file.stem + ".pdf")
    try:
        doc = fitz.open(n2file[file.stem])
    except:
        print(f"Can't open the pdf file")
        continue
    for index, i in enumerate(lines):
        if i in lines_check:
            continue
        try:
            page_pdf = doc.load_page((i-1))
            pix = page_pdf.get_pixmap()
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            size_in_bytes = len(img.tobytes())
            size_in_mb = size_in_bytes / (1024 * 1024)
            limit_mb = 4
            if size_in_mb > limit_mb:
                ratio = math.sqrt(limit_mb / size_in_mb)
                new_width = int(img.width * ratio)
                new_height = int(img.height * ratio)
                img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)

                new_size_mb = len(img.tobytes()) / (1024 * 1024)
                print(f"ย่อรูปเหลือ: {new_width}x{new_height} ({new_size_mb:.2f} MB)")
            
            img_byte_arr = io.BytesIO()
            img.save(img_byte_arr, format='JPEG', quality=90)
            img_byte_arr.seek(0) 

            file_payload = ('image.jpg', img_byte_arr, 'image/jpeg')

            result = extract_text_from_image(file_payload, api_key, model, task_type, max_tokens, temperature, top_p, repetition_penalty, pages=None)
            if result:
                if "5.18" in file.stem or "5-18" in file.stem:
                    with open(f"ocr-result/{path_dir}/{file.stem}_หน่วยที่_{index+1}.txt", "a", encoding="utf-8") as fr:
                        fr.write(f"Page:{i}\n")
                        fr.write(result)
                        fr.write("\n\n")
                else:
                    with open(f"ocr-result/{path_dir}/{file.stem}.txt", "a", encoding="utf-8") as fr:
                        fr.write(f"Page:{i}\n")
                        fr.write(result)
                        fr.write("\n\n")
                with open(f"ocr-success-page/{path_dir}/{file.stem}.txt", "a", encoding="utf-8") as fw:
                    fw.write(f"{i}\n")
                if index == 0: 
                    with open("except_ocr.txt", "a", encoding="utf-8") as fe:
                        fe.write(f"{file.stem}.txt\n")     
                print(f"Success on file: {file.stem} page: {i}")
                time.sleep(3)
        except Exception as e:
            print(f"Page: {i} not found in {file.stem + ".txt"}")
            print(e)
            break

Success on file: ชุดที่ 13 5-17 (บช) page: 1
Success on file: ชุดที่ 13 5-17 (บช) page: 2
Success on file: ชุดที่ 13 5-17 (บช) page: 3
Success on file: ชุดที่ 13 5-17 page: 1
ย่อรูปเหลือ: 936x1492 (4.00 MB)
Success on file: ชุดที่ 2 ส.ส. 5-17 (บช) page: 1
ย่อรูปเหลือ: 926x1508 (4.00 MB)
Success on file: ชุดที่ 2 ส.ส. 5-17 (บช) page: 2
ย่อรูปเหลือ: 936x1493 (4.00 MB)
Success on file: ชุดที่ 2 ส.ส. 5-17 (บช) page: 3
Success on file: ชุดที่ 2 ส.ส. 5-17 page: 1
Success on file: ชุดที่ 3 ส.ส. 5-17 (บช) page: 1
Success on file: ชุดที่ 3 ส.ส. 5-17 (บช) page: 2
Success on file: ชุดที่ 3 ส.ส. 5-17 (บช) page: 3
Success on file: ชุดที่ 3 ส.ส. 5-17 page: 1
Success on file: ชุดที่ 4 ส.ส. 5-17(บช) page: 1
Success on file: ชุดที่ 4 ส.ส. 5-17(บช) page: 2
Success on file: ชุดที่ 4 ส.ส. 5-17(บช) page: 3
Success on file: ชุดที่ 4 ส.ส. 5-17 page: 1
Success on file: ชุดที่ 5 ส.ส. 5-17(บช) page: 1
Success on file: ชุดที่ 5 ส.ส. 5-17(บช) page: 2
Success on file: ชุดที่ 5 ส.ส. 5-17(บช) page: 3
Success on file

KeyboardInterrupt: 